## **IF25-40304 Pembelajaran Mesin Multimodal · Kelompok 05**


## 0 · Dependensi (sekali)

In [1]:
# !pip install sentence-transformers numpy pandas


## 1 · Konfigurasi dan templat

In [2]:
import json, time, os, numpy as np

SEED  = 0
N_SUB = 500   # subset train untuk seleksi greedy
MAX_K = 5     # batas templat greedy

TEMPLATES_ID = [
    "{}", "foto {}", "sebuah foto {}", "gambar {}", "foto budaya Indonesia: {}",
    "dokumentasi budaya Indonesia: {}", "{} di Indonesia", "sebuah foto yang menunjukkan {}",
    "potret {}", "ini adalah foto {}",
]
TEMPLATES_EN = [
    "{}", "a photo of {}", "a picture of {}", "a photo showing {}",
    "a photo of Indonesian culture: {}", "documentation of Indonesian culture: {}",
    "{} in Indonesia", "this is a photo of {}",
]
print(f"Templat ID: {len(TEMPLATES_ID)} | Templat EN: {len(TEMPLATES_EN)} | seed={SEED}")


Templat ID: 10 | Templat EN: 8 | seed=0


## 2 · Utilitas dan protokol ranking (SAMA PERSIS dengan M1)

In [3]:
def l2norm(x):
    return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), 1e-12)

def apply_template(tpl, text, lower_first=True):
    if lower_first and tpl != "{}" and text and text[0].isupper() and not text[:2].isupper():
        text = text[0].lower() + text[1:]
    return tpl.format(text)

def encode_ensemble(encoder, texts, templates, batch_size=64, lower_first=True):
    """Rata-rata embedding teks di bawah tiap templat, lalu normalisasi ulang."""
    outs = []
    for tpl in templates:
        wrapped = [apply_template(tpl, t, lower_first) for t in texts]
        outs.append(np.asarray(encoder.encode(wrapped, batch_size=batch_size, normalize_embeddings=True)))
    return l2norm(np.mean(outs, axis=0))

def ranks_of_truth(Q, G):
    """Protokol M1 persis. Seri dihitung MENGUNTUNGKAN query (optimistis):
    peringkat = 1 + jumlah galeri yang skornya STRICTLY lebih tinggi dari pasangan benar."""
    S = Q @ G.T
    diag = np.diag(S)[:, None]
    return 1 + (S > diag).sum(axis=1)

def metrics(rank, k_map=10):
    r = {f"R@{k}": float((rank <= k).mean()) for k in (1, 5, 10)}
    r[f"mAP@{k_map}"] = float(np.where(rank <= k_map, 1.0 / rank, 0.0).mean())
    r["median_rank"] = float(np.median(rank))
    r["n_query"] = int(len(rank))
    return r

def four_directions(E, Ti, Te, k_map=10):
    rk = {
        "gambar->teks ID": ranks_of_truth(E, Ti),
        "gambar->teks EN": ranks_of_truth(E, Te),
        "teks ID->gambar": ranks_of_truth(Ti, E),
        "teks EN->gambar": ranks_of_truth(Te, E),
    }
    return {d: metrics(r, k_map) for d, r in rk.items()}

def timed(fn, *args, **kw):
    t = time.perf_counter(); out = fn(*args, **kw)
    return out, round(time.perf_counter() - t, 2)

# Nilai referensi M1 (results_val.json, split val) untuk uji sanity
M1_REF_R1 = {"gambar->teks ID": 0.214545, "gambar->teks EN": 0.447273,
             "teks ID->gambar": 0.196364, "teks EN->gambar": 0.381818}
print("Utilitas siap.")


Utilitas siap.


## 3 · Path dan load cache (sesuaikan)

Path di bawah memakai lokasi Drive yang sudah terbukti jalan.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/MULTIMODAL/Milestone/cache"                    # <-- sesuaikan
DATA_DIR  = "/content/drive/MyDrive/MULTIMODAL/bundle-mahasiswa/bundle-mahasiswa"  # <-- folder train.csv/val.csv

def load(split):
    z = np.load(f"{CACHE_DIR}/emb_{split}.npz")
    return {"img": z["img"], "cap_id": z["id"], "cap_en": z["en"]}

def captions(split):
    df = pd.read_csv(f"{DATA_DIR}/{split}.csv")
    return list(df["caption_id"]), list(df["caption_en"])

tr = load("train"); va = load("val")
cap_id_train, cap_en_train = captions("train")
cap_id_val, cap_en_val = captions("val")

# Pemeriksaan jumlah baris (cache harus sejajar dengan CSV)
assert len(cap_id_train) == len(tr["img"]) == len(tr["cap_id"]) == len(tr["cap_en"]), "train tidak sejajar"
assert len(cap_id_val) == len(va["img"]) == len(va["cap_id"]) == len(va["cap_en"]), "val tidak sejajar"
print("train:", len(cap_id_train), "| val:", len(cap_id_val), "| jumlah baris cocok")


Mounted at /content/drive
train: 2072 | val: 275 | jumlah baris cocok


## 4 · Load encoder (dicatat terpisah)

In [5]:
from sentence_transformers import SentenceTransformer

(img_enc, txt_id), t_load = timed(lambda: (
    SentenceTransformer("sentence-transformers/clip-ViT-B-32"),
    SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1"),
))
print(f"Encoder siap ({t_load} dtk)")


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

0_CLIPModel/model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

0_CLIPModel/model.safetensors: downloading bytes:           |  0.00B            

... (19 outputs left out)

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 1.57MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

Encoder siap (23.66 dtk)


## 5 · Pemeriksaan isi cache vs CSV

Encode 3 caption pertama lalu bandingkan dengan baris cache. Kosinus harus mendekati 1. Kalau tidak, urutan baris tidak cocok.

In [6]:
samp = [0, 1, 2]
e_id = np.asarray(txt_id.encode([cap_id_train[i] for i in samp], normalize_embeddings=True))
e_en = np.asarray(img_enc.encode([cap_en_train[i] for i in samp], normalize_embeddings=True))
cos_id = (e_id * tr["cap_id"][samp]).sum(1)
cos_en = (e_en * tr["cap_en"][samp]).sum(1)
print("kosinus caption ID vs cache:", np.round(cos_id, 4))
print("kosinus caption EN vs cache:", np.round(cos_en, 4))
assert (cos_id > 0.98).all() and (cos_en > 0.98).all(), "cache tidak cocok dengan CSV"
print("Urutan baris cache cocok dengan CSV (caption).")


kosinus caption ID vs cache: [1. 1. 1.]
kosinus caption EN vs cache: [1. 1. 1.]
Urutan baris cache cocok dengan CSV (caption).


## 6 · Seleksi greedy V1 (hanya train)

Waktu seleksi disimpan ke `m2_templates_terpilih.json`.

In [7]:
def greedy_select(encoder, texts_train, img_train, candidates, max_k=5, batch_size=64, tol=0.0):
    per = {t: np.asarray(encoder.encode([apply_template(t, x) for x in texts_train],
                                        batch_size=batch_size, normalize_embeddings=True))
           for t in candidates}
    def score(sel):
        Ti = l2norm(np.mean([per[t] for t in sel], axis=0))
        r1_ab = (ranks_of_truth(img_train, Ti) <= 1).mean()
        r1_ba = (ranks_of_truth(Ti, img_train) <= 1).mean()
        return (r1_ab + r1_ba) / 2
    chosen = ["{}"]
    trace = [(list(chosen), score(chosen))]
    best = trace[0][1]
    while len(chosen) < max_k:
        gains = [(score(chosen + [t]), t) for t in candidates if t not in chosen]
        if not gains: break
        s, t = max(gains)
        if s - best <= tol: break
        chosen.append(t); best = s; trace.append((list(chosen), s))
    return chosen, trace

rng = np.random.default_rng(SEED)
idx = np.sort(rng.choice(len(cap_id_train), size=min(N_SUB, len(cap_id_train)), replace=False))

(chosen_v1, trace), t_select = timed(greedy_select, txt_id,
                                     [cap_id_train[i] for i in idx], tr["img"][idx],
                                     TEMPLATES_ID, max_k=MAX_K)
print(f"Seleksi selesai ({t_select} dtk) | templat V1: {chosen_v1}")
for s, r in trace:
    print(f"  {s} -> R@1 train rata-rata dua arah ID = {r:.4f}")

json.dump({"seed": SEED, "n_sub": int(len(idx)), "split": "train", "chosen": chosen_v1,
           "protokol_ranking": "M1 (seri optimistis)", "waktu_seleksi_dtk": t_select,
           "trace": [{"set": s, "R@1_train_rata2_dua_arah_ID": r} for s, r in trace]},
          open("m2_templates_terpilih.json", "w"), ensure_ascii=False, indent=2)
print("Disimpan: m2_templates_terpilih.json")


Seleksi selesai (347.76 dtk) | templat V1: ['{}', 'ini adalah foto {}']
  ['{}'] -> R@1 train rata-rata dua arah ID = 0.1580
  ['{}', 'ini adalah foto {}'] -> R@1 train rata-rata dua arah ID = 0.1720
Disimpan: m2_templates_terpilih.json


## 7 · Diagnostik train (tidak untuk memilih apa pun)

Catatan: V1 dipilih dari subset train ini, jadi angka V1 di sini bias ke atas. Cell ini hanya untuk melihat apakah urutan V1 vs V2 sama di train.

In [8]:
Ti_tr_v1, _ = timed(encode_ensemble, txt_id, cap_id_train, chosen_v1)
Ti_tr_v2, _ = timed(encode_ensemble, txt_id, cap_id_train, TEMPLATES_ID)
diag_tr = {
    "V0": four_directions(tr["img"], tr["cap_id"], tr["cap_en"]),
    "V1": four_directions(tr["img"], Ti_tr_v1, tr["cap_en"]),
    "V2": four_directions(tr["img"], Ti_tr_v2, tr["cap_en"]),
}
for k, v in diag_tr.items():
    print(f"{k} train  G->ID R@1={v['gambar->teks ID']['R@1']:.4f}  ID->G R@1={v['teks ID->gambar']['R@1']:.4f}")


V0 train  G->ID R@1=0.0893  ID->G R@1=0.0671
V1 train  G->ID R@1=0.0941  ID->G R@1=0.0695
V2 train  G->ID R@1=0.1028  ID->G R@1=0.0690


## 8 · Evaluasi val (sekali, konfigurasi sudah dikunci) dengan biaya per tahap

In [9]:
biaya = {}
Ti_v1, biaya["V1_encode_dtk"] = timed(encode_ensemble, txt_id, cap_id_val, chosen_v1)
Ti_v2, biaya["V2_encode_dtk"] = timed(encode_ensemble, txt_id, cap_id_val, TEMPLATES_ID)
Te_k1, biaya["K1_encode_dtk"] = timed(encode_ensemble, img_enc, cap_en_val, TEMPLATES_EN[:len(chosen_v1)])

res_v0, biaya["V0_metrik_dtk"] = timed(four_directions, va["img"], va["cap_id"], va["cap_en"])
res_v1, biaya["V1_metrik_dtk"] = timed(four_directions, va["img"], Ti_v1, va["cap_en"])
res_v2, biaya["V2_metrik_dtk"] = timed(four_directions, va["img"], Ti_v2, va["cap_en"])
res_k1, biaya["K1_metrik_dtk"] = timed(four_directions, va["img"], va["cap_id"], Te_k1)

# Sanity: V0 harus sama dengan angka M1
for d, ref in M1_REF_R1.items():
    got = res_v0[d]["R@1"]
    print(f"{d:18s} V0={got:.6f}  M1={ref:.6f}  selisih={got-ref:+.6f}")
    assert abs(got - ref) < 1e-5, f"V0 tidak sama dengan M1 di {d}"
print("Sanity V0 = M1: OK")

n_cap = len(cap_id_val)
jumlah_encode = {
    "V1": len(chosen_v1) * n_cap, "V2": len(TEMPLATES_ID) * n_cap, "K1": len(chosen_v1) * n_cap,
}
results = {
    "V0 baseline M1": res_v0,
    "V1 ensemble ID (greedy, UTAMA)": res_v1,
    "V2 ensemble ID (semua 10, EKSPLORASI)": res_v2,
    "K1 kontrol ensemble EN": res_k1,
    "_meta": {
        "split": "val", "seed": SEED, "protokol_ranking": "M1 (seri optimistis)",
        "templat_V1": chosen_v1, "templat_V2": TEMPLATES_ID,
        "status": {"V0": "baseline M1", "V1": "utama (seleksi train)",
                   "V2": "eksplorasi (dirancang setelah melihat val V1)", "K1": "kontrol"},
        "biaya_dtk": {**biaya, "seleksi_V1_dtk": t_select, "load_encoder_dtk": t_load},
        "jumlah_encode_teks": jumlah_encode,
        "parameter_dilatih": 0,
        "runtime": "isi: CPU / GPU T4 (catat dari Runtime > Change runtime type)",
    },
}
json.dump(results, open("m2_metrik_val.json", "w"), ensure_ascii=False, indent=2)
print("Disimpan: m2_metrik_val.json")
print(json.dumps(biaya, indent=2))


gambar->teks ID    V0=0.214545  M1=0.214545  selisih=+0.000000
gambar->teks EN    V0=0.447273  M1=0.447273  selisih=-0.000000
teks ID->gambar    V0=0.196364  M1=0.196364  selisih=-0.000000
teks EN->gambar    V0=0.381818  M1=0.381818  selisih=+0.000000
Sanity V0 = M1: OK
Disimpan: m2_metrik_val.json
{
  "V1_encode_dtk": 38.19,
  "V2_encode_dtk": 187.67,
  "K1_encode_dtk": 31.27,
  "V0_metrik_dtk": 0.01,
  "V1_metrik_dtk": 0.01,
  "V2_metrik_dtk": 0.01,
  "K1_metrik_dtk": 0.01
}


## 9 · Tabel ringkas (selisih terhadap V0)

In [10]:
res = json.load(open("m2_metrik_val.json"))
base = res["V0 baseline M1"]
arah = ["teks ID->gambar", "gambar->teks ID", "teks EN->gambar", "gambar->teks EN"]
print("Varian\tarah\tR@1\tR@5\tR@10\tmAP@10\tdelta R@1 vs V0")
for v in [k for k in res if not k.startswith("_")]:
    for a in arah:
        m = res[v][a]
        print(f"{v}\t{a}\t{m['R@1']:.4f}\t{m['R@5']:.4f}\t{m['R@10']:.4f}\t{m['mAP@10']:.4f}\t{m['R@1']-base[a]['R@1']:+.4f}")


Varian	arah	R@1	R@5	R@10	mAP@10	delta R@1 vs V0
V0 baseline M1	teks ID->gambar	0.1964	0.4182	0.5055	0.2830	+0.0000
V0 baseline M1	gambar->teks ID	0.2145	0.3745	0.5055	0.2930	+0.0000
V0 baseline M1	teks EN->gambar	0.3818	0.6727	0.7709	0.5073	+0.0000
V0 baseline M1	gambar->teks EN	0.4473	0.7127	0.8327	0.5594	+0.0000
V1 ensemble ID (greedy, UTAMA)	teks ID->gambar	0.1891	0.4036	0.5200	0.2803	-0.0073
V1 ensemble ID (greedy, UTAMA)	gambar->teks ID	0.2436	0.4073	0.5236	0.3207	+0.0291
V1 ensemble ID (greedy, UTAMA)	teks EN->gambar	0.3818	0.6727	0.7709	0.5073	+0.0000
V1 ensemble ID (greedy, UTAMA)	gambar->teks EN	0.4473	0.7127	0.8327	0.5594	+0.0000
V2 ensemble ID (semua 10, EKSPLORASI)	teks ID->gambar	0.1964	0.4109	0.5236	0.2886	+0.0000
V2 ensemble ID (semua 10, EKSPLORASI)	gambar->teks ID	0.2291	0.4327	0.5600	0.3206	+0.0145
V2 ensemble ID (semua 10, EKSPLORASI)	teks EN->gambar	0.3818	0.6727	0.7709	0.5073	+0.0000
V2 ensemble ID (semua 10, EKSPLORASI)	gambar->teks EN	0.4473	0.7127	0.8327	0.5594	

## 10 · Holdout (opsional)

Hanya jalan kalau ada split test yang belum pernah dipakai untuk keputusan apa pun. Kalau tidak ada, cell ini hanya memberi tahu, dan laporan tidak boleh mengklaim konfirmasi independen.

In [11]:
test_csv = f"{DATA_DIR}/test.csv"
test_cache = f"{CACHE_DIR}/emb_test.npz"
if os.path.exists(test_csv) and os.path.exists(test_cache):
    te = load("test")
    cap_id_test, cap_en_test = captions("test")
    assert len(cap_id_test) == len(te["img"]), "test tidak sejajar"
    Ti_t1 = encode_ensemble(txt_id, cap_id_test, chosen_v1)
    out_test = {"V0": four_directions(te["img"], te["cap_id"], te["cap_en"]),
                "V1": four_directions(te["img"], Ti_t1, te["cap_en"])}
    json.dump(out_test, open("m2_metrik_test_holdout.json", "w"), ensure_ascii=False, indent=2)
    print("Holdout test dijalankan: V0 dan V1 saja (sekali).")
else:
    print("Tidak ada split test (test.csv / emb_test.npz). Tidak ada konfirmasi holdout.")
    print("Laporkan semua hasil sebagai evaluasi val, dan akui bahwa val sudah dipakai untuk dua varian.")


Tidak ada split test (test.csv / emb_test.npz). Tidak ada konfirmasi holdout.
Laporkan semua hasil sebagai evaluasi val, dan akui bahwa val sudah dipakai untuk dua varian.


In [13]:
from google.colab import files
files.download("m2_metrik_val.json")
files.download("m2_templates_terpilih.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>